#  Core Career Intelligence Engine — Matching & Recommendation System

## Overview
This notebook demonstrates the core intelligence engine for the **Career Intelligence Engine**:
1. **Candidate Profile Representation**: User skill ratings (0-4) and target roles.
2. **Baseline Skill Matching**: IDF-weighted skill overlap, match scoring, and skill gap identification.
3. **Role Matching**: Deriving dataset role profiles and matching candidate against target domains.
4. **Market Skill Demand**: Frequency analysis across 12,196 CS job postings.
5. **Learning Priority Recommendation Engine**: Transparent heuristic ranking next skills to learn.
6. **Machine Learning Experiments**: TF-IDF + Cosine Similarity & K-Means Skill Clustering.
7. **Evaluation**: Comparing Baseline vs ML.

### 1. Setup & Data Loading

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(os.path.join("..", ".."))

from ml.src.data.load_data import get_processed_data_dir
from ml.src.models.user_profile import get_default_sample_profile, UserProfile
from ml.src.models.matching import BaselineMatcher, RoleMatcher, TFIDFSimilarityMatcher, train_kmeans_skill_clusters
from ml.src.models.recommendation import MarketDemandAnalyzer, LearningPriorityEngine
from ml.src.evaluation.evaluate import evaluate_baseline_vs_ml

processed_dir = os.path.join("..", "data", "processed")
cs_df = pd.read_csv(os.path.join(processed_dir, "cs_job_postings.csv"))
matrix_df = pd.read_csv(os.path.join(processed_dir, "job_skill_matrix.csv"))
long_df = pd.read_csv(os.path.join(processed_dir, "job_skills_long.csv"))

print(f"CS Postings Shape: {cs_df.shape}")
print(f"Skill Matrix Shape: {matrix_df.shape}")
print(f"Long Skill Rows: {len(long_df)}")

### 2. Candidate User Profile Representation

In [ ]:
profile = get_default_sample_profile()
print(f"Candidate Name: {profile.candidate_name}")
print(f"Target Roles:   {profile.target_roles}")
print(f"Known Skills:   {sorted(list(profile.get_known_skills()))}")

### 3. Baseline Skill Overlap Matching & Job Ranking

In [ ]:
baseline_matcher = BaselineMatcher(matrix_df, cs_df, use_idf_weights=True)
top_jobs = baseline_matcher.rank_jobs(profile, top_n=10)
top_jobs[['job_id', 'title', 'company_name', 'match_score', 'matched_skills', 'missing_skills']].head(10)

### 4. Role Matching Analysis

In [ ]:
role_matcher = RoleMatcher(cs_df, long_df)
role_matches = role_matcher.match_user_to_roles(profile)
role_matches

### 5. Learning Priority Engine (Next Skill Recommendation)

In [ ]:
rec_engine = LearningPriorityEngine(cs_df, long_df)
rec_df, top_rec = rec_engine.recommend_next_skills(profile, top_n=10)
print(f"RECOMMENDED NEXT SKILL TO LEARN: {top_rec['recommended_skill']}")
print(top_rec['reason'])
rec_df[['skill', 'category', 'priority_score', 'demand_level', 'role_relevance_pct', 'market_demand_pct', 'explanation']]

### 6. Machine Learning: TF-IDF Cosine Similarity & K-Means Skill Clustering

In [ ]:
tfidf_matcher = TFIDFSimilarityMatcher(cs_df, max_features=1000)
tfidf_top_jobs = tfidf_matcher.rank_jobs(profile, top_n=10)
tfidf_top_jobs[['job_id', 'title', 'company_name', 'tfidf_match_score', 'extracted_skills']].head(10)

### 7. Baseline vs ML Evaluation Report

In [ ]:
eval_results = evaluate_baseline_vs_ml(profile, baseline_matcher, tfidf_matcher)
print(f"Baseline Speed: {eval_results['baseline_execution_time_ms']} ms")
print(f"TF-IDF Speed:   {eval_results['tfidf_execution_time_ms']} ms")
print(f"Spearman Correlation: {eval_results['spearman_rank_correlation']}")
print(f"Top-K Overlaps: {eval_results['top_k_ranking_overlap']}")